# Generator danych - Sieć Warsztatów Samochodowych i Sklepów z Akcesoriami

**Scenariusz biznesowy:** Sieć 100 warsztatów samochodowych i sklepów z akcesoriami w całej Polsce.  
**Okres:** 2020-01 do 2024-12 (5 lat)  
**Skala:** ~10-50 GB (konfigurowalna przez SCALE_FACTOR)  

## Tabele:
### Wymiarowe
1. `dim_locations` - lokalizacje (100)
2. `dim_employees` - pracownicy (~2000)
3. `dim_customers` - klienci (~500K)
4. `dim_vehicles` - pojazdy (~600K)
5. `dim_products` - produkty/części (~15K)
6. `dim_services` - katalog usług (~200)
7. `dim_suppliers` - dostawcy (~300)

### Faktowe
8. `fact_work_orders` - zlecenia warsztatowe (~5M)
9. `fact_work_order_items` - pozycje zleceń (~15M)
10. `fact_sales_transactions` - transakcje sklepowe (~30M)
11. `fact_sales_items` - pozycje sprzedaży (~90M)
12. `fact_invoices` - faktury (~35M)
13. `fact_payments` - płatności (~35M)
14. `fact_inventory_movements` - ruchy magazynowe (~50M)

### Wspierające
15. `fact_appointments` - rezerwacje (~5M)
16. `fact_purchase_orders` - zamówienia do dostawców (~500K)
17. `fact_purchase_order_items` - pozycje zamówień (~2M)
18. `fact_customer_feedback` - opinie (~2M)
19. `fact_loyalty_program` - program lojalnościowy (~500K)
20. `fact_employee_schedules` - grafiki pracy (~3M)

In [0]:
# Instalacja zależności
!pip install pandas pyarrow faker tqdm

In [0]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta, date
from faker import Faker
from tqdm import tqdm
import random
import uuid
import gc
import json
from dicts.referalls_dict import *

fake = Faker('pl_PL')
Faker.seed(42)
np.random.seed(42)
random.seed(42)

print('Biblioteki załadowane OK')

In [0]:
# ============================================================
# WIDGETS  –  single-day mode for Auto Loader testing
# ============================================================
# SINGLE_DAY_MODE = True  → generate data for TARGET_DATE only.
#   Files land in the same volume paths, so Auto Loader
#   detects them as new arrivals automatically.
# SINGLE_DAY_MODE = False → full historical run (2020-2026)
# ============================================================
try:
    dbutils.widgets.dropdown(
        'SINGLE_DAY_MODE', 'False', ['False', 'True'],
        label='Single Day Mode  (Auto Loader test)'
    )
    dbutils.widgets.text(
        'TARGET_DATE', str(date.today()),
        label='Target Date  (YYYY-MM-DD, used when Single Day Mode = True)'
    )
    _SINGLE_DAY_MODE = dbutils.widgets.get('SINGLE_DAY_MODE') == 'True'
    _TARGET_DATE_STR  = dbutils.widgets.get('TARGET_DATE')
except NameError:
    # Running outside Databricks – fall back to full historical run
    _SINGLE_DAY_MODE = False
    _TARGET_DATE_STR  = str(date.today())

print(f'SINGLE_DAY_MODE = {_SINGLE_DAY_MODE}')
if _SINGLE_DAY_MODE:
    print(f'TARGET_DATE     = {_TARGET_DATE_STR}')


In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

# SCALE_FACTOR: 1.0 = full data (~30 GB), 0.1 = ~3 GB, 0.01 = ~300 MB for testing
SCALE_FACTOR = 1.0

# Output destination
# 'local'  -> all tables saved under ./output_data
# 'volume' -> saved directly to Databricks Volume paths (dim/fact split)
OUTPUT_DESTINATION = 'volume'

if OUTPUT_DESTINATION == 'local':
    OUTPUT_DIR_DIM  = './output_data'
    OUTPUT_DIR_FACT = './output_data'
elif OUTPUT_DESTINATION == 'volume':
    OUTPUT_DIR_DIM  = '/Volumes/fake_car_workshop_franchise_pl/dim/dim_parquet_files'
    OUTPUT_DIR_FACT = '/Volumes/fake_car_workshop_franchise_pl/fact/fact_parquet_files'

# Format: 'parquet' or 'csv'
OUTPUT_FORMAT = 'parquet'

# Data period (may be overridden below by SINGLE_DAY_MODE widget)
DATE_START = date(2020, 1, 1)
DATE_END = date(2026, 4, 30)

# ── Single-day override (Auto Loader testing) ─────────────────────────────
if _SINGLE_DAY_MODE:
    DATE_START = date.fromisoformat(_TARGET_DATE_STR)
    DATE_END   = date.fromisoformat(_TARGET_DATE_STR)
    print(f'⚡ SINGLE_DAY_MODE active → generating data for {DATE_START} only')
# ─────────────────────────────────────────────────────────────────────────

NUM_YEARS = max(DATE_END.year - DATE_START.year + 1, 1)

# Chunk size for writing (rows)
CHUNK_SIZE = 200_000

# Number of locations
NUM_LOCATIONS = 100

if OUTPUT_DESTINATION == 'volume':
    print('Creating output directories...')
    os.makedirs(OUTPUT_DIR_DIM, exist_ok=True)
    os.makedirs(OUTPUT_DIR_FACT, exist_ok=True)

print(f'SCALE_FACTOR       = {SCALE_FACTOR}')
print(f'OUTPUT_DESTINATION = {OUTPUT_DESTINATION}')
print(f'OUTPUT_DIR_DIM     = {OUTPUT_DIR_DIM}')
print(f'OUTPUT_DIR_FACT    = {OUTPUT_DIR_FACT}')
print(f'DATE_START         = {DATE_START}')
print(f'DATE_END           = {DATE_END}')
print(f'NUM_YEARS          = {NUM_YEARS}')
print(f'Estimated data size: ~{SCALE_FACTOR * NUM_YEARS / 7 * 30:.1f} GB')


In [0]:
# ============================================================
# HELPERS
# ============================================================

def _get_base_dir(table_name):
    """Returns the correct base directory for dim or fact tables."""
    return OUTPUT_DIR_DIM if table_name.startswith('dim_') else OUTPUT_DIR_FACT


def save_table(df, table_name, partition_cols=None):
    """Saves DataFrame as parquet or csv."""
    table_dir = os.path.join(_get_base_dir(table_name), table_name)
    os.makedirs(table_dir, exist_ok=True)

    if OUTPUT_FORMAT == 'parquet':
        if partition_cols:
            pq.write_to_dataset(
                pa.Table.from_pandas(df),
                root_path=table_dir,
                partition_cols=partition_cols
            )
        else:
            pq.write_table(
                pa.Table.from_pandas(df),
                os.path.join(table_dir, f'{table_name}.parquet')
            )
    else:
        df.to_csv(os.path.join(table_dir, f'{table_name}.csv'), index=False)

    size_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f'  ✓ {table_name}: {len(df):,} rows, ~{size_mb:.1f} MB in memory')


def save_table_chunked(generate_func, table_name, total_rows, partition_cols=None):
    """Generates and saves data in chunks to conserve RAM."""
    table_dir = os.path.join(_get_base_dir(table_name), table_name)
    os.makedirs(table_dir, exist_ok=True)

    rows_written = 0
    chunk_num = 0

    with tqdm(total=total_rows, desc=table_name) as pbar:
        while rows_written < total_rows:
            chunk_rows = min(CHUNK_SIZE, total_rows - rows_written)
            df_chunk = generate_func(chunk_rows, rows_written)

            if OUTPUT_FORMAT == 'parquet':
                if partition_cols:
                    pq.write_to_dataset(
                        pa.Table.from_pandas(df_chunk),
                        root_path=table_dir,
                        partition_cols=partition_cols
                    )
                else:
                    pq.write_table(
                        pa.Table.from_pandas(df_chunk),
                        os.path.join(table_dir, f'{table_name}_part{chunk_num:04d}.parquet')
                    )
            else:
                mode = 'w' if chunk_num == 0 else 'a'
                header = chunk_num == 0
                df_chunk.to_csv(
                    os.path.join(table_dir, f'{table_name}.csv'),
                    index=False, mode=mode, header=header
                )

            rows_written += chunk_rows
            chunk_num += 1
            pbar.update(chunk_rows)
            del df_chunk
            gc.collect()

    print(f'  ✓ {table_name}: {rows_written:,} rows in {chunk_num} chunks')


def random_dates(start, end, n):
    """Generates n random dates from range."""
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    delta = (end_ts - start_ts).days
    
    # Handle single-day mode: when start == end, delta is 0
    if delta <= 0:
        return pd.to_datetime([start_ts] * n)
    
    random_days = np.random.randint(0, delta, size=n)
    dates = start_ts + pd.to_timedelta(random_days, unit='D')
    return dates


def seasonal_dates(start, end, n):
    """Generates dates with seasonality - more in spring/autumn periods."""
    dates = random_dates(start, end, n)
    months = dates.month
    # Seasonal weights: more in March-April (tyre change) and October-November
    seasonal_weights = {1: 0.7, 2: 0.7, 3: 1.4, 4: 1.4, 5: 1.0, 6: 0.9,
                        7: 0.8, 8: 0.8, 9: 1.0, 10: 1.4, 11: 1.3, 12: 0.6}
    weights = np.array([seasonal_weights[m] for m in months])
    weights = weights / weights.sum()
    indices = np.random.choice(len(dates), size=n, replace=True, p=weights)
    return dates[indices]


def generate_uuid_batch(n):
    """Generates a batch of UUIDs."""
    return [str(uuid.uuid4()) for _ in range(n)]


print('Helpers loaded OK')

## 1. Dane referencyjne (słowniki)

In [0]:

print(f'Załadowano słowniki:')
print(f'  - {len(MIASTA)} miast')
print(f'  - {len(MARKI_SAMOCHODOW)} marek samochodów')
print(f'  - {sum(len(v) for v in KATEGORIE_PRODUKTOW.values())} produktów w {len(KATEGORIE_PRODUKTOW)} kategoriach')
print(f'  - {len(KATALOG_USLUG)} usług warsztatowych')

## 2. Tabele wymiarowe (dimension tables)

In [0]:
# ============================================================
# dim_locations - 100 lokalizacji warsztatów/sklepów
# ============================================================
if _SINGLE_DAY_MODE == False:
    print('Generowanie dim_locations...')

    locations = []
    for i, (miasto, woj, lat, lon) in enumerate(MIASTA[:NUM_LOCATIONS]):
        typ = np.random.choice(TYPY_LOKALIZACJI, p=TYP_LOKALIZACJI_WAGI)
        otwarcie = fake.date_between(start_date=date(2005, 1, 1), end_date = DATE_START)
        locations.append({
            'location_id': i + 1,
            'location_code': f'LOC-{i+1:03d}',
            'nazwa': f'AutoSerwis {miasto}',
            'typ': typ,
            'ulica': fake.street_address(),
            'miasto': miasto,
            'wojewodztwo': woj,
            'kod_pocztowy': fake.postcode(),
            'latitude': lat + np.random.uniform(-0.02, 0.02),
            'longitude': lon + np.random.uniform(-0.02, 0.02),
            'telefon': fake.phone_number(),
            'email': f'serwis.{miasto.lower().replace(" ", "").replace("-", "")}@autoserwis.pl',
            'kierownik_id': None,  # uzupełnimy po generowaniu pracowników
            'liczba_stanowisk': np.random.randint(4, 12) if typ != 'sklep' else 0,
            'powierzchnia_m2': np.random.randint(200, 800),
            'data_otwarcia': otwarcie,
            'czy_aktywna': True if i < 95 else False,  # 5 lokalizacji zamkniętych
        })

    df_locations = pd.DataFrame(locations)
    save_table(df_locations, 'dim_locations')
    df_locations.head()

In [0]:
# ============================================================
# dim_employees - pracownicy (~20 na lokalizację = ~2000)
# ============================================================
if _SINGLE_DAY_MODE == False:
    print('Generowanie dim_employees...')

    employees = []
    emp_id = 1

    for _, loc in df_locations.iterrows():
        loc_id = loc['location_id']
        typ = loc['typ']
        
        # Kadra zarządzająca - zawsze
        for stanowisko in STANOWISKA['zarzadzanie']:
            employees.append({
                'employee_id': emp_id,
                'employee_code': f'EMP-{emp_id:05d}',
                'imie': fake.first_name(),
                'nazwisko': fake.last_name(),
                'pesel': fake.pesel(),
                'stanowisko': stanowisko,
                'location_id': loc_id,
                'data_zatrudnienia': fake.date_between(
                    start_date=loc['data_otwarcia'],
                    end_date=min(loc['data_otwarcia'] + timedelta(days=365), DATE_END)
                ),
                'data_zwolnienia': None,
                'stawka_godzinowa': round(np.random.uniform(45, 80), 2),
                'czy_aktywny': loc['czy_aktywna'],
            })
            emp_id += 1
        
        # Pracownicy warsztatu
        if typ in ('warsztat', 'warsztat_i_sklep'):
            n_mechanikow = np.random.randint(5, 10)
            for _ in range(n_mechanikow):
                stanowisko = random.choice(STANOWISKA['warsztat'])
                employees.append({
                    'employee_id': emp_id,
                    'employee_code': f'EMP-{emp_id:05d}',
                    'imie': fake.first_name_male() if random.random() < 0.9 else fake.first_name_female(),
                    'nazwisko': fake.last_name(),
                    'pesel': fake.pesel(),
                    'stanowisko': stanowisko,
                    'location_id': loc_id,
                    'data_zatrudnienia': fake.date_between(
                        start_date=loc['data_otwarcia'],
                        end_date=DATE_END
                    ),
                    'data_zwolnienia': fake.date_between(start_date=date(2022,1,1), end_date=DATE_END) if random.random() < 0.1 else None,
                    'stawka_godzinowa': round(np.random.uniform(30, 65), 2),
                    'czy_aktywny': random.random() > 0.1,
                })
                emp_id += 1
        
        # Pracownicy sklepu
        if typ in ('sklep', 'warsztat_i_sklep'):
            n_sprzedawcow = np.random.randint(3, 7)
            for _ in range(n_sprzedawcow):
                stanowisko = random.choice(STANOWISKA['sklep'])
                employees.append({
                    'employee_id': emp_id,
                    'employee_code': f'EMP-{emp_id:05d}',
                    'imie': fake.first_name(),
                    'nazwisko': fake.last_name(),
                    'pesel': fake.pesel(),
                    'stanowisko': stanowisko,
                    'location_id': loc_id,
                    'data_zatrudnienia': fake.date_between(
                        start_date=loc['data_otwarcia'],
                        end_date=DATE_END
                    ),
                    'data_zwolnienia': fake.date_between(start_date=date(2022,1,1), end_date=DATE_END) if random.random() < 0.15 else None,
                    'stawka_godzinowa': round(np.random.uniform(25, 45), 2),
                    'czy_aktywny': random.random() > 0.12,
                })
                emp_id += 1

    df_employees = pd.DataFrame(employees)
    save_table(df_employees, 'dim_employees')

    # Lista ID mechaników i sprzedawców do użycia w tabelach faktowych
    mechanic_ids = df_employees[df_employees['stanowisko'].isin(STANOWISKA['warsztat'])]['employee_id'].values
    seller_ids = df_employees[df_employees['stanowisko'].isin(STANOWISKA['sklep'])]['employee_id'].values

    # Mapowanie: location_id -> lista mechaników
    loc_mechanics = df_employees[df_employees['stanowisko'].isin(STANOWISKA['warsztat'])].groupby('location_id')['employee_id'].apply(list).to_dict()
    loc_sellers = df_employees[df_employees['stanowisko'].isin(STANOWISKA['sklep'])].groupby('location_id')['employee_id'].apply(list).to_dict()

    print(f'  Mechanicy: {len(mechanic_ids)}, Sprzedawcy: {len(seller_ids)}')
    df_employees.head()     

In [0]:
# ============================================================
# dim_customers - klienci (500K * SCALE_FACTOR)
# ============================================================
NUM_CUSTOMERS = int(random.randint(476_000, 689_000) * SCALE_FACTOR)
print(f'Generowanie dim_customers ({NUM_CUSTOMERS:,} klientów)...')

customer_types = np.random.choice(
    ['indywidualny', 'firma'], size=NUM_CUSTOMERS, p=[0.7, 0.3]
)

# Generate registration dates based on mode
if _SINGLE_DAY_MODE:
    registration_dates = [DATE_START] * NUM_CUSTOMERS
else:
    registration_dates = random_dates(DATE_START, DATE_END, NUM_CUSTOMERS)

customers = pd.DataFrame({
    'customer_id': np.arange(1, NUM_CUSTOMERS + 1),
    'customer_code': [f'CUS-{i:07d}' for i in range(1, NUM_CUSTOMERS + 1)],
    'typ_klienta': customer_types,
    'imie': [fake.first_name() if t == 'indywidualny' else '' for t in customer_types],
    'nazwisko': [fake.last_name() if t == 'indywidualny' else '' for t in customer_types],
    'nazwa_firmy': [fake.company() if t == 'firma' else '' for t in customer_types],
    'nip': [fake.company_vat() if t == 'firma' else '' for t in customer_types],
    'email': [fake.email() for _ in range(NUM_CUSTOMERS)],
    'telefon': [fake.phone_number() for _ in range(NUM_CUSTOMERS)],
    'miasto': np.random.choice([m[0] for m in MIASTA], size=NUM_CUSTOMERS),
    'kod_pocztowy': [fake.postcode() for _ in range(NUM_CUSTOMERS)],
    'data_rejestracji': registration_dates,
    'preferowana_lokalizacja_id': np.random.randint(1, NUM_LOCATIONS + 1, size=NUM_CUSTOMERS),
    'zgoda_marketing': np.random.choice([True, False], size=NUM_CUSTOMERS, p=[0.6, 0.4]),
})

df_customers = customers
save_table(df_customers, 'dim_customers')
customer_ids = df_customers['customer_id'].values
print(f'  Klienci indywidualni: {(df_customers["typ_klienta"]=="indywidualny").sum():,}')
print(f'  Klienci firmowi: {(df_customers["typ_klienta"]=="firma").sum():,}')
df_customers.head()

In [0]:
# ============================================================
# dim_vehicles - pojazdy klientów (600K * SCALE_FACTOR)
# ============================================================
NUM_VEHICLES = int(random.randint(489_000, 673_000) * SCALE_FACTOR)
print(f'Generowanie dim_vehicles ({NUM_VEHICLES:,} pojazdów)...')

marki = list(MARKI_SAMOCHODOW.keys())
wagi = [MARKA_WAGI[m] for m in marki]
wagi_norm = np.array(wagi) / sum(wagi)

chosen_marki = np.random.choice(marki, size=NUM_VEHICLES, p=wagi_norm)
chosen_modele = [random.choice(MARKI_SAMOCHODOW[m]) for m in chosen_marki]

vehicles = pd.DataFrame({
    'vehicle_id': np.arange(1, NUM_VEHICLES + 1),
    'customer_id': np.random.choice(customer_ids, size=NUM_VEHICLES),
    'marka': chosen_marki,
    'model': chosen_modele,
    'rocznik': np.random.randint(2005, 2025, size=NUM_VEHICLES),
    'vin': [fake.bothify('???#########??????').upper() for _ in range(NUM_VEHICLES)],
    'nr_rejestracyjny': [fake.license_plate() for _ in range(NUM_VEHICLES)],
    'typ_paliwa': np.random.choice(TYPY_PALIWA, size=NUM_VEHICLES, p=PALIWO_WAGI),
    'pojemnosc_silnika': np.random.choice(
        [1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.2, 2.5, 3.0],
        size=NUM_VEHICLES,
        p=[0.05, 0.10, 0.15, 0.12, 0.18, 0.12, 0.12, 0.06, 0.05, 0.05]
    ),
    'moc_km': np.random.randint(60, 350, size=NUM_VEHICLES),
    'kolor': np.random.choice(KOLORY, size=NUM_VEHICLES),
    'przebieg_km': np.random.randint(5000, 350000, size=NUM_VEHICLES),
    'data_pierwszej_rejestracji': random_dates(date(2005,1,1), DATE_END, NUM_VEHICLES),
})

df_vehicles = vehicles
save_table(df_vehicles, 'dim_vehicles')
vehicle_ids = df_vehicles['vehicle_id'].values
print(f'  Średnio {NUM_VEHICLES / NUM_CUSTOMERS:.1f} pojazdów na klienta')
df_vehicles.head()

In [0]:
# ============================================================
# dim_products - produkty/części (~15K z wariantami)
# ============================================================
if _SINGLE_DAY_MODE == False:
    print('Generowanie dim_products...')

    products = []
    prod_id = 1

    for kategoria, produkty_lista in KATEGORIE_PRODUKTOW.items():
        for nazwa_bazowa in produkty_lista:
            # Każdy produkt ma kilka wariantów (różne marki producenta)
            producenci = random.sample(
                ['Bosch', 'Continental', 'Valeo', 'Hella', 'Mann', 'Mahle', 'NGK',
                'Brembo', 'TRW', 'KYB', 'Monroe', 'Sachs', 'LuK', 'Gates',
                'SKF', 'Dayco', 'Castrol', 'Mobil', 'Shell', 'Total', 'Motul',
                'Liqui Moly', 'K2', 'Meguiars', 'Sonax', 'Goodyear', 'Michelin',
                'Continental', 'Bridgestone', 'Pirelli', 'Varta', 'Exide', 'Banner'],
                k=min(random.randint(2, 6), 32)
            )
            for producent in producenci:
                cena_bazowa = round(np.random.uniform(5, 800), 2)
                # Droższe produkty: opony, akumulatory, sprzęgło
                if 'Opona' in nazwa_bazowa:
                    cena_bazowa = round(np.random.uniform(180, 600), 2)
                elif 'Akumulator' in nazwa_bazowa:
                    cena_bazowa = round(np.random.uniform(250, 800), 2)
                elif 'Komplet sprzęgła' in nazwa_bazowa or 'Koło dwumasowe' in nazwa_bazowa:
                    cena_bazowa = round(np.random.uniform(400, 2000), 2)
                elif 'Amortyzator' in nazwa_bazowa:
                    cena_bazowa = round(np.random.uniform(100, 400), 2)
                elif 'Filtr' in nazwa_bazowa:
                    cena_bazowa = round(np.random.uniform(15, 80), 2)
                elif 'Klocki' in nazwa_bazowa or 'Tarcze' in nazwa_bazowa:
                    cena_bazowa = round(np.random.uniform(60, 300), 2)
                elif 'Olej' in nazwa_bazowa:
                    cena_bazowa = round(np.random.uniform(30, 180), 2)
                elif 'Żarówka' in nazwa_bazowa:
                    cena_bazowa = round(np.random.uniform(8, 120), 2)
                
                marza = round(np.random.uniform(1.15, 1.45), 2)
                
                products.append({
                    'product_id': prod_id,
                    'product_code': f'PRD-{prod_id:06d}',
                    'nazwa': f'{nazwa_bazowa} {producent}',
                    'kategoria': kategoria,
                    'producent': producent,
                    'cena_zakupu_netto': cena_bazowa,
                    'cena_sprzedazy_netto': round(cena_bazowa * marza, 2),
                    'vat_procent': 23,
                    'jednostka': 'szt' if 'Olej' not in nazwa_bazowa and 'Płyn' not in nazwa_bazowa else 'L',
                    'waga_kg': round(np.random.uniform(0.1, 15), 2),
                    'min_stan_magazynowy': np.random.randint(2, 20),
                    'czy_aktywny': random.random() > 0.05,
                })
                prod_id += 1

    df_products = pd.DataFrame(products)
    save_table(df_products, 'dim_products')
    product_ids = df_products['product_id'].values
    print(f'  Produktów: {len(df_products):,} w {len(KATEGORIE_PRODUKTOW)} kategoriach')
    df_products.head()

In [0]:
# ============================================================
# dim_services - katalog usług warsztatowych
# ============================================================
if _SINGLE_DAY_MODE == False:
    print('Generowanie dim_services...')

    services = []
    for i, (nazwa, kategoria, min_c, max_c, czas) in enumerate(KATALOG_USLUG):
        services.append({
            'service_id': i + 1,
            'service_code': f'SRV-{i+1:03d}',
            'nazwa': nazwa,
            'kategoria': kategoria,
            'cena_min_netto': min_c,
            'cena_max_netto': max_c,
            'szacowany_czas_min': czas,
            'czy_aktywna': True,
        })

    df_services = pd.DataFrame(services)
    save_table(df_services, 'dim_services')
    service_ids = df_services['service_id'].values

    # ============================================================
    # dim_suppliers - dostawcy części
    # ============================================================
    print('Generowanie dim_suppliers...')

    NUM_SUPPLIERS = 300
    suppliers = []
    for i in range(NUM_SUPPLIERS):
        suppliers.append({
            'supplier_id': i + 1,
            'supplier_code': f'SUP-{i+1:04d}',
            'nazwa': fake.company(),
            'nip': fake.company_vat(),
            'miasto': random.choice([m[0] for m in MIASTA]),
            'adres': fake.street_address(),
            'kod_pocztowy': fake.postcode(),
            'telefon': fake.phone_number(),
            'email': fake.company_email(),
            'osoba_kontaktowa': fake.name(),
            'warunki_platnosci_dni': random.choice([14, 21, 30, 45, 60]),
            'min_wartosc_zamowienia': round(np.random.uniform(200, 2000), 2),
            'czy_aktywny': random.random() > 0.08,
        })

    df_suppliers = pd.DataFrame(suppliers)
    save_table(df_suppliers, 'dim_suppliers')
    supplier_ids = df_suppliers['supplier_id'].values

    print(f'=== PODSUMOWANIE TABEL WYMIAROWYCH ===')
    for name, df in [('dim_locations', df_locations), ('dim_employees', df_employees),
                    ('dim_customers', df_customers), ('dim_vehicles', df_vehicles),
                    ('dim_products', df_products), ('dim_services', df_services),
                    ('dim_suppliers', df_suppliers)]:
        print(f'  {name}: {len(df):,} wierszy')

## 3. Tabele faktowe - zlecenia warsztatowe

In [0]:
# ============================================================
# fact_work_orders - zlecenia warsztatowe (~700K/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_WORK_ORDERS = int(random.randint(389_000,716_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_work_orders ({NUM_WORK_ORDERS:,} zleceń)...')

# Lokalizacje z warsztatem
workshop_locs = df_locations[df_locations['typ'].isin(['warsztat', 'warsztat_i_sklep'])]['location_id'].values

def generate_work_orders_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    loc_ids = np.random.choice(workshop_locs, size=chunk_size)
    
    # Przypisz mechanika z danej lokalizacji
    mech_ids = []
    for lid in loc_ids:
        mechs = loc_mechanics.get(lid, mechanic_ids[:5])
        mech_ids.append(random.choice(mechs))
    
    return pd.DataFrame({
        'work_order_id': np.arange(offset + 1, offset + chunk_size + 1),
        'work_order_code': [f'WO-{i:08d}' for i in range(offset + 1, offset + chunk_size + 1)],
        'location_id': loc_ids,
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'vehicle_id': np.random.choice(vehicle_ids, size=chunk_size),
        'mechanic_id': mech_ids,
        'data_przyjecia': dates,
        'data_zakonczenia': dates + pd.to_timedelta(np.random.randint(0, 5, size=chunk_size), unit='D'),
        'status': np.random.choice(STATUSY_ZLECEN, size=chunk_size, p=STATUSY_WAGI),
        'przebieg_przy_przyjęciu': np.random.randint(10000, 350000, size=chunk_size),
        'uwagi_klienta': np.random.choice(
            ['', 'Stuk przy hamowaniu', 'Silnik traci moc', 'Wyciek oleju',
             'Wymiana opon sezonowa', 'Przegląd okresowy', 'Klima nie chłodzi',
             'Kontrolka silnika', 'Hałas z zawieszenia', 'Wymiana klocków',
             'Przygotowanie do przeglądu', 'Wymiana oleju', 'Problem z rozrusznikiem',
             'Drgania kierownicy', 'Wymiana świec', ''],
            size=chunk_size
        ),
        'rok': dates.year,
        'miesiac': dates.month,
    })

save_table_chunked(generate_work_orders_chunk, 'fact_work_orders', NUM_WORK_ORDERS, 
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_work_order_items - pozycje zleceń (~2.1M/rok * NUM_YEARS * SCALE_FACTOR)
# Każde zlecenie ma 1-6 pozycji (usługa + ewentualnie części)
# ============================================================
NUM_WO_ITEMS = int(random.randint(1_675_000,2_317_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_work_order_items ({NUM_WO_ITEMS:,} pozycji)...')

def generate_wo_items_chunk(chunk_size, offset):
    wo_ids = np.random.randint(1, NUM_WORK_ORDERS + 1, size=chunk_size)
    # Losowy typ pozycji: usługa lub część
    typ_pozycji = np.random.choice(['usluga', 'czesc'], size=chunk_size, p=[0.4, 0.6])
    
    srv_ids = np.where(
        typ_pozycji == 'usluga',
        np.random.choice(service_ids, size=chunk_size),
        0
    )
    prod_ids = np.where(
        typ_pozycji == 'czesc',
        np.random.choice(product_ids, size=chunk_size),
        0
    )
    
    ilosc = np.where(typ_pozycji == 'usluga', 1, np.random.randint(1, 5, size=chunk_size))
    cena_netto = np.where(
        typ_pozycji == 'usluga',
        np.random.uniform(30, 2000, size=chunk_size),
        np.random.uniform(5, 500, size=chunk_size)
    )
    cena_netto = np.round(cena_netto, 2)
    
    return pd.DataFrame({
        'wo_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'work_order_id': wo_ids,
        'typ_pozycji': typ_pozycji,
        'service_id': srv_ids.astype(int),
        'product_id': prod_ids.astype(int),
        'ilosc': ilosc,
        'cena_jednostkowa_netto': cena_netto,
        'wartosc_netto': np.round(cena_netto * ilosc, 2),
        'vat_procent': 23,
        'wartosc_brutto': np.round(cena_netto * ilosc * 1.23, 2),
        'rabat_procent': np.random.choice([0, 0, 0, 5, 10, 15], size=chunk_size),
    })

save_table_chunked(generate_wo_items_chunk, 'fact_work_order_items', NUM_WO_ITEMS)

## 4. Tabele faktowe - sprzedaż sklepowa

In [0]:
# ============================================================
# fact_sales_transactions - transakcje sprzedaży sklepowej (~4.3M/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_SALES = int(random.randint(2_952_00, 4_456_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_sales_transactions ({NUM_SALES:,} transakcji)...')

# Lokalizacje ze sklepem
shop_locs = df_locations[df_locations['typ'].isin(['sklep', 'warsztat_i_sklep'])]['location_id'].values

def generate_sales_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    hours = np.random.choice(range(7, 20), size=chunk_size, 
                              p=[0.03, 0.08, 0.10, 0.10, 0.09, 0.08, 0.08,
                                 0.08, 0.08, 0.08, 0.08, 0.07, 0.05])
    minutes = np.random.randint(0, 60, size=chunk_size)
    
    timestamps = dates + pd.to_timedelta(hours, unit='h') + pd.to_timedelta(minutes, unit='m')
    loc_ids = np.random.choice(shop_locs, size=chunk_size)
    
    seller_arr = []
    for lid in loc_ids:
        sellers = loc_sellers.get(lid, seller_ids[:3])
        seller_arr.append(random.choice(sellers))
    
    # ~70% transakcji ma klienta zarejestrowanego, ~30% to walk-in
    has_customer = np.random.random(size=chunk_size) < 0.7
    cust_ids = np.where(has_customer, np.random.choice(customer_ids, size=chunk_size), 0)
    
    return pd.DataFrame({
        'transaction_id': np.arange(offset + 1, offset + chunk_size + 1),
        'transaction_code': [f'TRX-{i:09d}' for i in range(offset + 1, offset + chunk_size + 1)],
        'location_id': loc_ids,
        'customer_id': cust_ids,
        'employee_id': seller_arr,
        'data_transakcji': timestamps,
        'metoda_platnosci': np.random.choice(METODY_PLATNOSCI, size=chunk_size, p=PLATNOSC_WAGI),
        'nr_paragonu': [f'PAR/{random.randint(1,999):03d}/{i+offset+1:08d}' for i in range(chunk_size)],
        'rok': dates.year,
        'miesiac': dates.month,
    })

save_table_chunked(generate_sales_chunk, 'fact_sales_transactions', NUM_SALES,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_sales_items - pozycje sprzedaży (20-28M/rok * NUM_YEARS * SCALE_FACTOR)
# Średnio 3 pozycje na transakcję
# ============================================================
NUM_SALES_ITEMS = int(random.randint(20_000_000,28_000_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_sales_items ({NUM_SALES_ITEMS:,} pozycji)...')

def generate_sales_items_chunk(chunk_size, offset):
    trx_ids = np.random.randint(1, NUM_SALES + 1, size=chunk_size)
    prod_ids_chunk = np.random.choice(product_ids, size=chunk_size)
    ilosc = np.random.choice([1, 1, 1, 2, 2, 3, 4], size=chunk_size)
    cena_netto = np.round(np.random.uniform(3, 600, size=chunk_size), 2)
    rabat = np.random.choice([0, 0, 0, 0, 5, 10, 15, 20], size=chunk_size)
    wartosc_po_rabacie = np.round(cena_netto * ilosc * (1 - rabat/100), 2)
    
    return pd.DataFrame({
        'sales_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'transaction_id': trx_ids,
        'product_id': prod_ids_chunk,
        'ilosc': ilosc,
        'cena_jednostkowa_netto': cena_netto,
        'rabat_procent': rabat,
        'wartosc_netto': wartosc_po_rabacie,
        'vat_procent': 23,
        'wartosc_brutto': np.round(wartosc_po_rabacie * 1.23, 2),
    })

save_table_chunked(generate_sales_items_chunk, 'fact_sales_items', NUM_SALES_ITEMS)

## 5. Tabele faktowe - faktury, płatności, magazyn

In [0]:
# ============================================================
# fact_invoices - faktury (~5M/rok * NUM_YEARS * SCALE_FACTOR)
# Faktury powiązane z work_orders i sales_transactions
# ============================================================
NUM_INVOICES = int(random.randint(3_145_00, 5_000_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_invoices ({NUM_INVOICES:,} faktur)...')

def generate_invoices_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    
    # ~15% faktur to faktury za zlecenia warsztatowe, ~85% za sprzedaż sklepową
    source_type = np.random.choice(
        ['work_order', 'sales'], size=chunk_size, p=[0.15, 0.85]
    )
    source_ids = np.where(
        source_type == 'work_order',
        np.random.randint(1, max(NUM_WORK_ORDERS, 1) + 1, size=chunk_size),
        np.random.randint(1, max(NUM_SALES, 1) + 1, size=chunk_size)
    )
    
    wartosc_netto = np.round(np.random.lognormal(mean=4.5, sigma=1.0, size=chunk_size), 2)
    wartosc_netto = np.clip(wartosc_netto, 10, 50000)
    wartosc_vat = np.round(wartosc_netto * 0.23, 2)
    
    # Typ: faktura VAT, paragon, faktura korygująca
    typ_dokumentu = np.random.choice(
        ['faktura_VAT', 'paragon', 'faktura_korygujaca'],
        size=chunk_size, p=[0.35, 0.60, 0.05]
    )
    
    return pd.DataFrame({
        'invoice_id': np.arange(offset + 1, offset + chunk_size + 1),
        'invoice_code': [f'FV/{dates[i].year}/{i+offset+1:08d}' for i in range(chunk_size)],
        'typ_dokumentu': typ_dokumentu,
        'source_type': source_type,
        'source_id': source_ids,
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'data_wystawienia': dates,
        'data_sprzedazy': dates - pd.to_timedelta(np.random.randint(0, 3, size=chunk_size), unit='D'),
        'termin_platnosci': dates + pd.to_timedelta(
            np.random.choice([0, 7, 14, 30], size=chunk_size, p=[0.5, 0.15, 0.2, 0.15]), unit='D'
        ),
        'wartosc_netto': wartosc_netto,
        'wartosc_vat': wartosc_vat,
        'wartosc_brutto': np.round(wartosc_netto + wartosc_vat, 2),
        'status': np.random.choice(
            ['oplacona', 'oczekuje', 'przeterminowana', 'anulowana'],
            size=chunk_size, p=[0.80, 0.10, 0.07, 0.03]
        ),
        'rok': dates.year,
        'miesiac': dates.month,
    })

save_table_chunked(generate_invoices_chunk, 'fact_invoices', NUM_INVOICES,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_payments - płatności (~5M/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_PAYMENTS = int(random.randint(3_145_00, 5_000_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_payments ({NUM_PAYMENTS:,} płatności)...')

def generate_payments_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    kwota = np.round(np.random.lognormal(mean=4.5, sigma=1.0, size=chunk_size), 2)
    kwota = np.clip(kwota, 5, 60000)
    
    return pd.DataFrame({
        'payment_id': np.arange(offset + 1, offset + chunk_size + 1),
        'invoice_id': np.random.randint(1, max(NUM_INVOICES, 1) + 1, size=chunk_size),
        'data_platnosci': dates,
        'kwota': kwota,
        'metoda_platnosci': np.random.choice(METODY_PLATNOSCI, size=chunk_size, p=PLATNOSC_WAGI),
        'status': np.random.choice(
            ['zrealizowana', 'oczekuje', 'odrzucona', 'zwrot'],
            size=chunk_size, p=[0.90, 0.05, 0.03, 0.02]
        ),
        'numer_transakcji': [f'PAY-{uuid.uuid4().hex[:12].upper()}' for _ in range(chunk_size)],
        'rok': dates.year,
        'miesiac': dates.month,
    })

save_table_chunked(generate_payments_chunk, 'fact_payments', NUM_PAYMENTS,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_inventory_movements - ruchy magazynowe (~7M/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_INVENTORY = int(random.randint(5_744_00, 7_289_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_inventory_movements ({NUM_INVENTORY:,} ruchów)...')

def generate_inventory_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    
    typ_ruchu = np.random.choice(
        ['przyjecie', 'wydanie_sprzedaz', 'wydanie_warsztat', 'zwrot', 'korekta', 'inwentaryzacja'],
        size=chunk_size, p=[0.25, 0.35, 0.25, 0.05, 0.05, 0.05]
    )
    
    ilosc = np.random.randint(1, 20, size=chunk_size)
    # Wydania mają ujemną ilość
    ilosc = np.where(
        np.isin(typ_ruchu, ['wydanie_sprzedaz', 'wydanie_warsztat']),
        -ilosc, ilosc
    )
    
    return pd.DataFrame({
        'movement_id': np.arange(offset + 1, offset + chunk_size + 1),
        'product_id': np.random.choice(product_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'typ_ruchu': typ_ruchu,
        'ilosc': ilosc,
        'data_ruchu': dates,
        'dokument_zrodlowy': np.random.choice(
            ['PZ', 'WZ', 'RW', 'ZW', 'KOR', 'INW'],
            size=chunk_size
        ),
        'nr_dokumentu': [f'DOC-{i+offset+1:09d}' for i in range(chunk_size)],
        'wartosc_netto': np.round(np.abs(ilosc) * np.random.uniform(5, 500, size=chunk_size), 2),
        'uwagi': np.random.choice(['', '', '', 'Dostawa regularna', 'Zamówienie specjalne',
                                     'Zwrot od klienta', 'Korekta stanów', ''], size=chunk_size),
        'rok': dates.year,
        'miesiac': dates.month,
    })

save_table_chunked(generate_inventory_chunk, 'fact_inventory_movements', NUM_INVENTORY,
                   partition_cols=['rok', 'miesiac'])

## 6. Tabele wspierające

In [0]:
# ============================================================
# fact_appointments - rezerwacje wizyt (~700K/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_APPOINTMENTS = int(random.randint(345_00, 881_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_appointments ({NUM_APPOINTMENTS:,} rezerwacji)...')

def generate_appointments_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    hours = np.random.choice(range(7, 17), size=chunk_size)
    timestamps = dates + pd.to_timedelta(hours, unit='h')
    
    return pd.DataFrame({
        'appointment_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'vehicle_id': np.random.choice(vehicle_ids, size=chunk_size),
        'location_id': np.random.choice(workshop_locs, size=chunk_size),
        'service_id': np.random.choice(service_ids, size=chunk_size),
        'data_rezerwacji': dates - pd.to_timedelta(np.random.randint(1, 14, size=chunk_size), unit='D'),
        'data_wizyty': timestamps,
        'status': np.random.choice(
            ['potwierdzona', 'zrealizowana', 'anulowana', 'niestawienie_sie'],
            size=chunk_size, p=[0.10, 0.75, 0.10, 0.05]
        ),
        'kanal_rezerwacji': np.random.choice(
            ['telefon', 'online', 'osobiscie', 'email'],
            size=chunk_size, p=[0.35, 0.40, 0.15, 0.10]
        ),
        'uwagi': np.random.choice(
            ['', '', '', 'Proszę o kontakt telefoniczny', 'Samochód zastępczy',
             'Preferuję rano', 'Pilne', 'Umówiony wcześniej', ''],
            size=chunk_size
        ),
        'rok': dates.year,
        'miesiac': dates.month,
    })

save_table_chunked(generate_appointments_chunk, 'fact_appointments', NUM_APPOINTMENTS,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_purchase_orders - zamówienia do dostawców (~70K/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_PO = int(random.randint(47_000, 85_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_purchase_orders ({NUM_PO:,} zamówień)...')

def generate_po_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    wartosc = np.round(np.random.lognormal(mean=7, sigma=0.8, size=chunk_size), 2)
    wartosc = np.clip(wartosc, 200, 100000)
    
    return pd.DataFrame({
        'po_id': np.arange(offset + 1, offset + chunk_size + 1),
        'po_code': [f'PO-{i+offset+1:07d}' for i in range(chunk_size)],
        'supplier_id': np.random.choice(supplier_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'data_zamowienia': dates,
        'data_dostawy_planowana': dates + pd.to_timedelta(np.random.randint(3, 21, size=chunk_size), unit='D'),
        'data_dostawy_rzeczywista': dates + pd.to_timedelta(np.random.randint(3, 25, size=chunk_size), unit='D'),
        'wartosc_netto': wartosc,
        'wartosc_brutto': np.round(wartosc * 1.23, 2),
        'status': np.random.choice(
            ['zlozono', 'w_realizacji', 'dostarczone', 'czesciowo_dostarczone', 'anulowane'],
            size=chunk_size, p=[0.03, 0.05, 0.85, 0.05, 0.02]
        ),
        'rok': dates.year,
    })

save_table_chunked(generate_po_chunk, 'fact_purchase_orders', NUM_PO)

# ============================================================
# fact_purchase_order_items - pozycje zamówień (~285K/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_PO_ITEMS = int(random.randint(245_00, 316_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_purchase_order_items ({NUM_PO_ITEMS:,} pozycji)...')

def generate_po_items_chunk(chunk_size, offset):
    ilosc = np.random.randint(1, 50, size=chunk_size)
    cena = np.round(np.random.uniform(5, 500, size=chunk_size), 2)
    
    return pd.DataFrame({
        'po_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'po_id': np.random.randint(1, max(NUM_PO, 1) + 1, size=chunk_size),
        'product_id': np.random.choice(product_ids, size=chunk_size),
        'ilosc_zamowiona': ilosc,
        'ilosc_dostarczona': np.clip(ilosc + np.random.randint(-2, 1, size=chunk_size), 0, 100),
        'cena_jednostkowa_netto': cena,
        'wartosc_netto': np.round(cena * ilosc, 2),
    })

save_table_chunked(generate_po_items_chunk, 'fact_purchase_order_items', NUM_PO_ITEMS)

In [0]:
# ============================================================
# fact_customer_feedback - opinie klientów (~285K/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_FEEDBACK = int(random.randint(245_00, 316_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_customer_feedback ({NUM_FEEDBACK:,} opinii)...')

KOMENTARZE = [
    'Bardzo profesjonalna obsługa', 'Szybka realizacja', 'Polecam!',
    'Trochę za drogo', 'Długi czas oczekiwania', 'Świetna komunikacja',
    'Fachowa naprawa', 'Samochód gotowy przed terminem', 'Miła obsługa',
    'Mogłoby być taniej', 'Wrócę na pewno', 'Solidna robota',
    'Uczciwe ceny', 'Problem wrócił po miesiącu', 'Brak uwag',
    'Rewelacja!', 'Przeciętnie', 'Do poprawy', 'OK', '',
]

def generate_feedback_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    
    # Oceny z rozkładem: więcej pozytywnych
    oceny = np.random.choice(
        [1, 2, 3, 4, 5], size=chunk_size,
        p=[0.03, 0.05, 0.12, 0.30, 0.50]
    )
    
    return pd.DataFrame({
        'feedback_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'work_order_id': np.random.randint(1, max(NUM_WORK_ORDERS, 1) + 1, size=chunk_size),
        'data_opinii': dates,
        'ocena': oceny,
        'komentarz': np.random.choice(KOMENTARZE, size=chunk_size),
        'kategoria': np.random.choice(
            ['obsluga', 'jakosc_naprawy', 'czas_realizacji', 'cena', 'czystosc', 'ogolna'],
            size=chunk_size, p=[0.20, 0.25, 0.15, 0.15, 0.10, 0.15]
        ),
        'kanal': np.random.choice(
            ['google', 'formularz_online', 'email', 'telefon'],
            size=chunk_size, p=[0.40, 0.30, 0.20, 0.10]
        ),
    })

save_table_chunked(generate_feedback_chunk, 'fact_customer_feedback', NUM_FEEDBACK)

In [0]:
# ============================================================
# fact_loyalty_program - program lojalnościowy (500K * SCALE_FACTOR)
# ============================================================
NUM_LOYALTY = int(random.randint(345_00, 516_000) * SCALE_FACTOR)
print(f'Generowanie fact_loyalty_program ({NUM_LOYALTY:,} wpisów)...')

def generate_loyalty_chunk(chunk_size, offset):
    dates = random_dates(date(2021, 1, 1), DATE_END, chunk_size)  # Program od 2021
    
    return pd.DataFrame({
        'loyalty_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'data_zdarzenia': dates,
        'typ_zdarzenia': np.random.choice(
            ['naliczenie_punktow', 'wymiana_punktow', 'bonus', 'wygasniecie'],
            size=chunk_size, p=[0.60, 0.20, 0.10, 0.10]
        ),
        'punkty': np.random.choice(
            [-500, -200, -100, 10, 20, 50, 100, 200, 500],
            size=chunk_size, p=[0.05, 0.07, 0.08, 0.20, 0.25, 0.15, 0.10, 0.05, 0.05]
        ),
        'opis': np.random.choice(
            ['Zakup w sklepie', 'Usługa warsztatowa', 'Bonus powitalny',
             'Bonus urodzinowy', 'Wymiana na rabat 10%', 'Wymiana na rabat 20%',
             'Wymiana na darmowy przegląd', 'Punkty wygasłe', 'Polecenie znajomego'],
            size=chunk_size
        ),
        'saldo_po': np.random.randint(0, 5000, size=chunk_size),
        'poziom': np.random.choice(
            ['standard', 'silver', 'gold', 'platinum'],
            size=chunk_size, p=[0.50, 0.30, 0.15, 0.05]
        ),
    })

save_table_chunked(generate_loyalty_chunk, 'fact_loyalty_program', NUM_LOYALTY)

In [0]:
# ============================================================
# fact_employee_schedules - grafiki pracy (~430K/rok * NUM_YEARS * SCALE_FACTOR)
# ============================================================
NUM_SCHEDULES = int(random.randint(345_00, 476_000) * NUM_YEARS * SCALE_FACTOR)
print(f'Generowanie fact_employee_schedules ({NUM_SCHEDULES:,} wpisów)...')

all_employee_ids = df_employees['employee_id'].values

def generate_schedules_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    
    godzina_start = np.random.choice([6, 7, 8, 9, 10, 12, 14], size=chunk_size,
                                       p=[0.05, 0.25, 0.30, 0.15, 0.05, 0.10, 0.10])
    czas_pracy = np.random.choice([4, 6, 8, 10, 12], size=chunk_size,
                                    p=[0.10, 0.10, 0.60, 0.15, 0.05])
    
    return pd.DataFrame({
        'schedule_id': np.arange(offset + 1, offset + chunk_size + 1),
        'employee_id': np.random.choice(all_employee_ids, size=chunk_size),
        'data': dates,
        'godzina_start': godzina_start,
        'godzina_koniec': godzina_start + czas_pracy,
        'typ_zmiany': np.random.choice(
            ['dzienna', 'poranna', 'popoludniowa', 'nocna', 'wolne', 'urlop', 'chorobowe'],
            size=chunk_size, p=[0.40, 0.15, 0.15, 0.02, 0.15, 0.08, 0.05]
        ),
        'nadgodziny_h': np.random.choice(
            [0, 0, 0, 0, 0, 1, 2, 3, 4],
            size=chunk_size
        ),
        'obecnosc': np.random.choice(
            ['obecny', 'nieobecny_usprawiedliwiony', 'nieobecny_nieusprawiedliwiony', 'spozniony'],
            size=chunk_size, p=[0.88, 0.08, 0.02, 0.02]
        ),
    })

save_table_chunked(generate_schedules_chunk, 'fact_employee_schedules', NUM_SCHEDULES)

## 7. Walidacja i statystyki

In [0]:
# ============================================================
# VALIDATION AND SUMMARY
# ============================================================
import glob as glob_module

print('=' * 60)
print('DATA GENERATION SUMMARY')
print('=' * 60)
print(f'SCALE_FACTOR:       {SCALE_FACTOR}')
print(f'OUTPUT_DESTINATION: {OUTPUT_DESTINATION}')
print(f'OUTPUT_DIR_DIM:     {OUTPUT_DIR_DIM}')
print(f'OUTPUT_DIR_FACT:    {OUTPUT_DIR_FACT}')
print()

total_size = 0
table_stats = []

# deduplicated in case both dirs are the same (local mode)
scan_dirs = list(dict.fromkeys([OUTPUT_DIR_DIM, OUTPUT_DIR_FACT]))

for base_dir in scan_dirs:
    if not os.path.isdir(base_dir):
        continue
    for table_name in sorted(os.listdir(base_dir)):
        table_path = os.path.join(base_dir, table_name)
        if os.path.isdir(table_path):
            # Count file sizes
            size = 0
            file_count = 0
            for root, dirs, files in os.walk(table_path):
                for f in files:
                    fp = os.path.join(root, f)
                    size += os.path.getsize(fp)
                    file_count += 1

            size_mb = size / (1024 * 1024)
            total_size += size
            table_stats.append({
                'table': table_name,
                'files': file_count,
                'size_MB': round(size_mb, 1),
            })

df_stats = pd.DataFrame(table_stats)
print(df_stats.to_string(index=False))
print()
print(f'TOTAL SIZE: {total_size / (1024**3):.2f} GB')
print(f'Estimated size at SCALE_FACTOR=1.0: ~{total_size / (1024**3) / SCALE_FACTOR:.1f} GB')
print()
print('Done! Output locations:')
print(f'  DIM:  {OUTPUT_DIR_DIM}')
print(f'  FACT: {OUTPUT_DIR_FACT}')
print()
print('To load data into Databricks:')
print('  1. Upload the output_data/ directory to DBFS or Unity Catalog Volume')
print('  2. Use spark.read.parquet("dbfs:/path/to/table/")')
print('  3. Or CREATE TABLE ... USING PARQUET LOCATION ...')